# DPF-Nutrition Food2K

Frozen monocular depth prediction, RGB-D CAB fusion, and five-task direct nutrition regression initialized with Food2K ResNet101 weights.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/nutrition5k'
REPO = '/content/Nutrition5k'
DATA = f'{DRIVE}/data'
CONFIG = f'{REPO}/configs/colab/dpf_nutrition_food2k.yaml'
CHECKPOINT_DIR = f'{DRIVE}/checkpoints/dpf_food2k'
RESULT_DIR = f'{DRIVE}/results/dpf_food2k'
FOOD2K_FILE_ID = '1_xM2qv1NIjev8voYjXLhfnxDzvFNB85q'
FOOD2K_URL = f'https://drive.google.com/file/d/{FOOD2K_FILE_ID}/view'


In [ ]:
import os, subprocess, sys

if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/T0MYYY/Nutrition5k.git', REPO, '-b', 'dpf', '-q'], check=True)
%cd /content/Nutrition5k
subprocess.run(['git', 'pull', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', 'transformers', 'accelerate', 'tqdm', 'gdown'], check=True)
subprocess.run(['apt-get', 'install', '-qq', '-y', 'zstd'], check=True)
if REPO not in sys.path:
    sys.path.insert(0, REPO)


In [ ]:
import os, subprocess, sys, yaml

with open(CONFIG) as f:
    food2k_cfg = yaml.safe_load(f)
FOOD2K_WEIGHTS = food2k_cfg['model'].get('food2k_resnet101_path', '')
if not FOOD2K_WEIGHTS:
    raise ValueError('model.food2k_resnet101_path is empty')

os.makedirs(os.path.dirname(FOOD2K_WEIGHTS), exist_ok=True)
if not os.path.isfile(FOOD2K_WEIGHTS):
    subprocess.run([sys.executable, '-m', 'gdown', '--fuzzy', FOOD2K_URL, '-O', FOOD2K_WEIGHTS], check=True)
print({'food2k_weights': FOOD2K_WEIGHTS, 'bytes': os.path.getsize(FOOD2K_WEIGHTS)})


In [ ]:
import json, os, platform, shutil, subprocess, textwrap

probe_code = r'''
import json, os, platform, shutil, subprocess, torch
info = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'drive_exists': os.path.isdir('/content/drive/MyDrive/nutrition5k'),
    'dev_shm': shutil.disk_usage('/dev/shm')._asdict(),
    'ram_kb': open('/proc/meminfo').read().splitlines()[:3],
}
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    info.update({'gpu': p.name, 'vram_gb': round(p.total_memory / 1024**3, 2)})
print(json.dumps(info, indent=2))
'''

run_url = os.environ.get('COLAB_RUN_URL', '').rstrip('/')
if run_url:
    import requests
    response = requests.post(f'{run_url}/run', json={'code': probe_code}, timeout=120)
    response.raise_for_status()
    print(response.text)
else:
    exec(probe_code)

In [ ]:
import os, subprocess

os.makedirs('/dev/shm/realsense_overhead', exist_ok=True)
os.makedirs('/dev/shm/pred_depth', exist_ok=True)

def read_ids(split_txt):
    return [line.strip() for line in open(split_txt) if line.strip()]

def missing_rgb_ids(split_txt):
    return [dish_id for dish_id in read_ids(split_txt)
            if not os.path.isfile(f'/dev/shm/realsense_overhead/{dish_id}/rgb.png')]

MISSING_OVERHEAD = {}

def extract_if_needed(archive, marker, split_txt):
    missing = missing_rgb_ids(split_txt)
    if not missing:
        print(f'{archive}: ready ({len(read_ids(split_txt))} rgb files)')
        MISSING_OVERHEAD[archive] = []
        return []
    src = f'{DATA}/{archive}'
    print(f'Extracting {archive}; missing before extract: {len(missing)}')
    subprocess.run(f'zstd -d --stdout "{src}" | tar -x -C /dev/shm', shell=True, check=True)
    missing = missing_rgb_ids(split_txt)
    if missing:
        print(f'WARNING: {archive} is missing {len(missing)} split RGB files, e.g. {missing[:5]}')
    MISSING_OVERHEAD[archive] = missing
    return missing

jobs = [
    ('depth_train.tar.zst', '/dev/shm/realsense_overhead', f'{DATA}/splits/depth_train_ids.txt'),
    ('depth_test.tar.zst', '/dev/shm/realsense_overhead', f'{DATA}/splits/depth_test_ids.txt'),
]
for job in jobs:
    extract_if_needed(*job)
print(sum(1 for d in os.scandir('/dev/shm/realsense_overhead') if d.is_dir()))

In [ ]:
import os, subprocess

def missing_pred_ids(split_txt):
    return [dish_id for dish_id in read_ids(split_txt)
            if not os.path.isfile(f'/dev/shm/pred_depth/{dish_id}/depth_pred.png')]

def extract_pred_if_needed(archive, split_txt):
    missing = missing_pred_ids(split_txt)
    if not missing:
        print(f'{archive}: ready ({len(read_ids(split_txt))} predicted depth files)')
        return True
    src = f'{DATA}/{archive}'
    if not os.path.isfile(src):
        return False
    print(f'Extracting {archive}; missing before extract: {len(missing)}')
    subprocess.run(f'zstd -d --stdout "{src}" | tar -x -C /dev/shm', shell=True, check=True)
    missing = missing_pred_ids(split_txt)
    if missing:
        print(f'{archive}: still missing {len(missing)} predicted depth files after extract')
    return not missing

have_train = extract_pred_if_needed('dpf_pred_depth_train.tar.zst', f'{DATA}/splits/depth_train_ids.txt')
have_test = extract_pred_if_needed('dpf_pred_depth_test.tar.zst', f'{DATA}/splits/depth_test_ids.txt')
print({'train_cache': have_train, 'test_cache': have_test})

In [ ]:
if not (have_train and have_test):
    sample_dir = '/dev/shm/dpf_depth_sample'
    subprocess.run([
        sys.executable, 'scripts/generate_dpf_depth_cache.py',
        '--overhead-root', '/dev/shm/realsense_overhead',
        '--split-ids', f'{DATA}/splits/depth_train_ids.txt',
        '--split', 'train',
        '--output-dir', sample_dir,
        '--max-images', '3',
        '--skip-archive',
    ], check=True)
    from PIL import Image
    first = next(os.scandir(f'{sample_dir}/pred_depth')).path
    png = os.path.join(first, 'depth_pred.png')
    img = Image.open(png)
    print({'mode': img.mode, 'size': img.size})

In [ ]:
CACHE_DIR = '/dev/shm/dpf_depth_cache'
os.makedirs(CACHE_DIR, exist_ok=True)
import torch
DPF_DEPTH_BATCH = 64 if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory >= 40 * 1024**3 else 16
print(f'DPF depth batch size: {DPF_DEPTH_BATCH}')

if not have_train:
    subprocess.run([
        sys.executable, 'scripts/generate_dpf_depth_cache.py',
        '--overhead-root', '/dev/shm/realsense_overhead',
        '--split-ids', f'{DATA}/splits/depth_train_ids.txt',
        '--split', 'train',
        '--output-dir', CACHE_DIR,
        '--archive-dir', DATA,
        '--batch-size', str(DPF_DEPTH_BATCH),
        '--allow-missing-rgb',
    ], check=True)
if not have_test:
    subprocess.run([
        sys.executable, 'scripts/generate_dpf_depth_cache.py',
        '--overhead-root', '/dev/shm/realsense_overhead',
        '--split-ids', f'{DATA}/splits/depth_test_ids.txt',
        '--split', 'test',
        '--output-dir', CACHE_DIR,
        '--archive-dir', DATA,
        '--batch-size', str(DPF_DEPTH_BATCH),
        '--allow-missing-rgb',
    ], check=True)
extract_pred_if_needed('dpf_pred_depth_train.tar.zst', f'{DATA}/splits/depth_train_ids.txt')
extract_pred_if_needed('dpf_pred_depth_test.tar.zst', f'{DATA}/splits/depth_test_ids.txt')

In [ ]:
from nutrition5k_pkg.train import run

best_ckpt = f'{CHECKPOINT_DIR}/best.pt'
if os.path.isfile(best_ckpt):
    print(best_ckpt)
else:
    best_ckpt = run(CONFIG)
    print(best_ckpt)

In [ ]:
import json, os, torch
from nutrition5k_pkg.evaluate import evaluate
from nutrition5k_pkg.models.dpf_nutrition import DPFNutritionNet
from nutrition5k_pkg.train import load_config

cfg = load_config(CONFIG)
model = DPFNutritionNet(
    tasks=cfg['model']['tasks'],
    pretrained=False,
    fusion_channels=cfg['model']['fusion_channels'],
    head_hidden=cfg['model']['head_hidden'],
)
state = torch.load(best_ckpt, map_location='cpu')
model.load_state_dict(state)
results = evaluate(model, CONFIG, RESULT_DIR)
os.makedirs(RESULT_DIR, exist_ok=True)
with open(f'{RESULT_DIR}/metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
mean_pmae = sum(v['mae_pct'] for v in results.values()) / len(results)
print({'mean_pmae': mean_pmae, 'targets': {'calories': 14.7, 'mass': 10.6, 'fat': 22.6, 'carb': 20.7, 'protein': 20.2, 'mean': 17.8}})

In [ ]:
RELEASE_RUNTIME = False
if RELEASE_RUNTIME:
    from google.colab import runtime
    runtime.unassign()